<a href="https://colab.research.google.com/github/arinamartynyuk/-4-9/blob/main/%D0%9B%D0%A04_%D0%9C%D0%B0%D1%80%D1%82%D0%B8%D0%BD%D1%8E%D0%BA_4_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота №4. Регресійні моделі: глибоке дослідження

**Датасет:** Car Price Prediction Multiple Linear Regression  
https://www.kaggle.com/datasets/hellbuoy/car-price-prediction?select=CarPrice_Assignment.csv

**Мета роботи:** виконати підготовку даних, побудувати та порівняти регресійні моделі для прогнозування ціни автомобіля, оцінити їх якість і проаналізувати отримані результати.

> У шаблоні наведено лише умови завдань. Реалізацію необхідно виконати самостійно у порожніх комірках.


## 1. Завантаження та первинний аналіз даних

Завантажити датасет та виконати його первинний огляд:
- визначити кількість об'єктів і ознак;
- переглянути назви стовпців;
- визначити типи даних;
- визначити цільову змінну;
- вивести основні статистичні характеристики даних.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
import io, os

if os.path.exists("CarPrice_Assignment.csv"):
    df = pd.read_csv("CarPrice_Assignment.csv")
else:
    uploaded = files.upload()
    csv_name = next((name for name in uploaded if name.lower().endswith(".csv")), None)
    if csv_name is None:
        raise ValueError("Завантажте CSV-файл датасету CarPrice_Assignment.csv")
    df = pd.read_csv(io.BytesIO(uploaded[csv_name]))

display(df.head())
print("Розмір датасету:", df.shape)
print("\\nНазви стовпців:", list(df.columns))
print("\\nТипи даних:")
display(df.dtypes.to_frame("Тип"))
print("\\nСтатистика числових даних:")
display(df.describe())
print("\\nЦільова змінна: price")

## 2. Перевірка якості даних

Перевірити:
- наявність пропущених значень;
- наявність повних дублікатів;
- кількість унікальних значень в ознаках.

Під час аналізу врахувати, що `car_ID` є унікальним ідентифікатором автомобіля і сам по собі не повинен використовуватися для визначення дублікатів за змістовими характеристиками.


In [ ]:
print("Пропущені значення:")
display(df.isna().sum().to_frame("Кількість пропусків"))
print("Повні дублікати (без car_ID):", df.drop(columns=["car_ID"], errors="ignore").duplicated().sum())
print("Унікальні значення в ознаках:")
display(df.nunique().to_frame("Кількість унікальних значень").sort_values(by="Кількість унікальних значень"))

## 3. Робота з car_ID

Зберегти `car_ID` окремо для подальшої ідентифікації автомобілів у результатах прогнозування.  
Не використовувати `car_ID` як ознаку під час навчання моделей.


In [ ]:
car_ids = df["car_ID"].copy()
data = df.drop(columns=["car_ID"]).copy()
print("car_ID збережено окремо; його не використовуватимемо як predictor.")
print("Форма даних після вилучення car_ID:", data.shape)

## 4. Дослідження розподілів числових ознак

Дослідити розподіли числових ознак:
- побудувати відповідні графіки;
- обчислити коефіцієнти асиметрії;
- визначити ознаки з найбільш вираженою асиметрією;
- коротко прокоментувати отримані результати.


In [ ]:
numeric_all = data.select_dtypes(include=np.number)
numeric_features = numeric_all.drop(columns=["price"], errors="ignore")
fig, axes = plt.subplots(nrows=max(1, int(np.ceil(len(numeric_features.columns)/3))), ncols=3, figsize=(15, 4*max(1, int(np.ceil(len(numeric_features.columns)/3)))))
axes = np.array(axes).reshape(-1)
for ax, col in zip(axes, numeric_features.columns):
    sns.histplot(data=numeric_features, x=col, kde=True, ax=ax)
    ax.set_title(col)
for ax in axes[len(numeric_features.columns):]:
    ax.remove()
plt.tight_layout()
plt.show()

skew_initial = numeric_features.skew().sort_values(key=lambda x: abs(x), ascending=False)
display(skew_initial.to_frame("Коефіцієнт асиметрії"))
print("Найбільш асиметричні ознаки:")
display(skew_initial.head(5))

## 5. Формування train/test вибірок

Відокремити цільову змінну `price` від ознак та розділити дані на:
- навчальну вибірку — 80%;
- тестову вибірку — 20%.

Забезпечити відтворюваність поділу.  
Окремо зберегти `car_ID` для об'єктів, які потрапили до train і test.

**Важливо:** усі подальші параметри передобробки даних визначати тільки за навчальною вибіркою.


In [ ]:
from sklearn.model_selection import train_test_split

X = data.drop(columns=["price"])
y = data["price"]
X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(
    X, y, car_ids, test_size=0.2, random_state=42
)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("ID train/test:", ids_train.shape, ids_test.shape)

## 6. Аналіз та обробка викидів

Для неперервних числових ознак навчальної вибірки:
- визначити потенційні викиди методом IQR;
- вивести кількість потенційних викидів для кожної ознаки;
- обґрунтувати спосіб їх обробки.

Для невеликого набору даних не видаляти автоматично всі рядки з викидами. Використати обмеження крайніх значень межами, визначеними за навчальною вибіркою.

Цільову змінну `price` не використовувати для визначення меж викидів.  
До тестової вибірки застосувати ті самі межі, які були отримані за train.


In [ ]:
# IQR-аналіз і clipping: межі визначаються тільки на train, без цільової змінної.
continuous_candidates = X_train.select_dtypes(include=np.number).columns.drop("symboling", errors="ignore")
outlier_counts = {}
clip_bounds = {}
for col in continuous_candidates:
    q1, q3 = X_train[col].quantile([0.25, 0.75])
    iqr = q3-q1
    lower, upper = q1-1.5*iqr, q3+1.5*iqr
    outlier_counts[col] = int(((X_train[col] < lower) | (X_train[col] > upper)).sum())
    clip_bounds[col] = (lower, upper)
display(pd.Series(outlier_counts, name="Потенційні викиди").sort_values(ascending=False).to_frame())

X_train_clip, X_test_clip = X_train.copy(), X_test.copy()
for col, (lower, upper) in clip_bounds.items():
    X_train_clip[col] = X_train_clip[col].clip(lower, upper)
    X_test_clip[col] = X_test_clip[col].clip(lower, upper)
print("Викиди не видалялися: крайні значення обмежено train-межами IQR. Це зберігає невеликий датасет.")

## 7. Аналіз кореляцій та відбір ознак

За навчальною вибіркою:
- побудувати матрицю кореляцій числових ознак;
- знайти пари ознак із сильною кореляцією, для яких `|r| > 0.85`;
- для кожної такої пари залишити одну ознаку;
- під час вибору врахувати силу зв'язку кожної ознаки з цільовою змінною `price`;
- вивести список сильно корельованих пар та перелік ознак, які видаляються.

Із тестової вибірки видалити ті самі ознаки.  
Не використовувати тестову вибірку для прийняття рішення про відбір ознак.


In [ ]:
# Кореляції та відбір ознак лише на train.
corr = X_train_clip.select_dtypes(include=np.number).corr()
plt.figure(figsize=(14, 10))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Кореляційна матриця числових ознак (train)")
plt.tight_layout()
plt.show()

numeric_train = X_train_clip.select_dtypes(include=np.number)
corr_full = numeric_train.corr()
pairs = []
drop_features = set()
cols = list(corr_full.columns)
for i in range(len(cols)):
    for j in range(i+1, len(cols)):
        a, b = cols[i], cols[j]
        r = corr_full.loc[a,b]
        if abs(r) > 0.85:
            # Зберігаємо ознаку, що має сильніший абсолютний зв'язок з price.
            ca = abs(X_train_clip[a].corr(y_train))
            cb = abs(X_train_clip[b].corr(y_train))
            removed = b if ca >= cb else a
            pairs.append((a,b,r,ca,cb,removed))
            drop_features.add(removed)
pairs_df = pd.DataFrame(pairs, columns=["Ознака 1","Ознака 2","Кореляція між ознаками","|corr з price| 1","|corr з price| 2","Видалена ознака"])
display(pairs_df if len(pairs_df) else pd.DataFrame(columns=pairs_df.columns))
print("Видалені ознаки:", sorted(drop_features))
X_train_selected = X_train_clip.drop(columns=list(drop_features), errors="ignore")
X_test_selected = X_test_clip.drop(columns=list(drop_features), errors="ignore")

## 8. Підготовка числових і категоріальних ознак

Розділити ознаки на:
- неперервні числові;
- дискретну числову ознаку `symboling`;
- категоріальні.

Для неперервних числових ознак виконати послідовно:
1. перетворення Yeo–Johnson для зменшення асиметрії;
2. масштабування за допомогою StandardScaler.

Для `symboling` виконати масштабування без Yeo–Johnson.

Категоріальні ознаки закодувати методом One-Hot Encoding. Передбачити коректну обробку категорій у test, яких не було у train.

Усі перетворення об'єднати в єдиний механізм передобробки даних, який навчається тільки на `X_train`.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import PowerTransformer, StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

continuous = X_train_selected.select_dtypes(include=np.number).columns.drop("symboling", errors="ignore").tolist()
discrete = ["symboling"] if "symboling" in X_train_selected.columns else []
categorical = X_train_selected.select_dtypes(exclude=np.number).columns.tolist()

continuous_pipeline = Pipeline([("yeo_johnson", PowerTransformer(method="yeo-johnson")), ("scaler", StandardScaler())])
discrete_pipeline = Pipeline([("scaler", StandardScaler())])
categorical_pipeline = OneHotEncoder(handle_unknown="ignore")

preprocessor = ColumnTransformer([
    ("continuous", continuous_pipeline, continuous),
    ("discrete", discrete_pipeline, discrete),
    ("categorical", categorical_pipeline, categorical)
], remainder="drop")

print("Неперервні:", continuous)
print("symboling:", discrete)
print("Категоріальні:", categorical)

## 9. Перевірка трансформації числових ознак

Перевірити результат трансформації неперервних числових ознак:
- застосувати до train ту саму послідовність перетворень, яка буде використана в моделях;
- повторно обчислити коефіцієнти асиметрії;
- порівняти їх із початковими значеннями;
- зробити короткий висновок щодо ефективності трансформації.


In [ ]:
# Трансформація та порівняння асиметрії на train.
transformed_cont = continuous_pipeline.fit_transform(X_train_selected[continuous])
skew_after = pd.DataFrame(transformed_cont, columns=continuous).skew()
skew_comparison = pd.DataFrame({"До трансформації": X_train_selected[continuous].skew(), "Після Yeo-Johnson + scaling": skew_after})
display(skew_comparison.sort_values(by="До трансформації", key=lambda s: abs(s), ascending=False))
print("Середня абсолютна асиметрія до:", X_train_selected[continuous].skew().abs().mean())
print("Середня абсолютна асиметрія після:", skew_after.abs().mean())

## 10. Побудова регресійних моделей

Побудувати та навчити такі моделі:
- Linear Regression;
- Ridge Regression;
- Lasso Regression;
- Random Forest Regressor;
- Gradient Boosting Regressor;
- Support Vector Regressor (SVR).

Для кожної моделі створити єдиний Pipeline, який містить:
1. попередню обробку ознак;
2. відповідну регресійну модель.

Використати однакову схему підготовки даних для коректного порівняння моделей.


In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Lasso Regression": Lasso(alpha=0.01, max_iter=10000),
    "Random Forest": RandomForestRegressor(n_estimators=300, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "SVR": SVR(kernel="rbf")
}
pipelines = {name: Pipeline([("preprocessing", preprocessor), ("model", model)]) for name, model in models.items()}
print("Створено", len(pipelines), "pipeline-моделей.")

## 11. Оцінювання якості моделей

Для кожної моделі:
- виконати навчання на тренувальній вибірці;
- отримати прогноз для тестової вибірки;
- обчислити метрики **R², MAE, MSE та MAPE**.

`MAPE` подати у відсотках.

Результати всіх моделей об'єднати в одну порівняльну таблицю та впорядкувати за значенням R² від найбільшого до найменшого.

Під час інтерпретації врахувати:
- R² — чим більше, тим краще;
- MAE, MSE, MAPE — чим менше, тим краще.


In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
results = []
predictions = {}
for name, pipe in pipelines.items():
    pipe.fit(X_train_selected, y_train)
    pred = pipe.predict(X_test_selected)
    predictions[name] = pred
    results.append({
        "Модель": name,
        "R²": r2_score(y_test, pred),
        "MAE": mean_absolute_error(y_test, pred),
        "MSE": mean_squared_error(y_test, pred),
        "MAPE (%)": np.mean(np.abs((y_test-pred)/y_test))*100
    })
results_df = pd.DataFrame(results).sort_values("R²", ascending=False).reset_index(drop=True)
display(results_df)
best_model_name = results_df.iloc[0]["Модель"]
print("Найкраща модель за R²:", best_model_name)

## 12. Пояснення вибору типу моделі

Пояснити, чому класифікаційна модель, наприклад Logistic Regression, не підходить для безпосереднього прогнозування неперервної ціни автомобіля.

Зазначити, до якого типу задачі належить прогнозування `price` та чому моделі, використані в роботі, відповідають цій задачі.


**Прогнозування ціни автомобіля є задачею регресії**, оскільки цільова змінна `price` є неперервною числовою величиною. Регресійні алгоритми оцінюють конкретне число, а не клас чи категорію.

Logistic Regression є класифікаційною моделлю: вона призначена для визначення класів (або ймовірностей класів). Без перетворення ціни на категорії вона не підходить для прямого прогнозування неперервної вартості. У роботі застосовані регресійні моделі, які відповідають типу задачі.

## 13. Візуальне порівняння прогнозів

Для кожної побудованої моделі зобразити співвідношення:
- фактичних значень `price`;
- прогнозованих значень `price`.

Додати лінію ідеального прогнозу та проаналізувати, наскільки близько до неї розташовані прогнозовані значення. Звернути увагу на великі відхилення.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, (name, pred) in zip(axes.flat, predictions.items()):
    ax.scatter(y_test, pred, alpha=0.7)
    min_val, max_val = min(y_test.min(), pred.min()), max(y_test.max(), pred.max())
    ax.plot([min_val, max_val], [min_val, max_val], linestyle="--")
    ax.set_title(name)
    ax.set_xlabel("Фактична price")
    ax.set_ylabel("Прогнозована price")
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 14. Прогноз для 10 автомобілів

Випадково вибрати 10 автомобілів із тестової вибірки.

За допомогою моделі Random Forest отримати для них прогноз ціни та сформувати таблицю з такими даними:
- `car_ID`;
- фактична ціна;
- прогнозована ціна;
- абсолютна помилка;
- абсолютна відсоткова помилка.

Прокоментувати отримані результати.


In [ ]:
rf = pipelines["Random Forest"]
sample_indices = X_test_selected.sample(n=min(10, len(X_test_selected)), random_state=42).index
sample_X = X_test_selected.loc[sample_indices]
sample_y = y_test.loc[sample_indices]
sample_ids = ids_test.loc[sample_indices]
sample_pred = rf.predict(sample_X)
sample_results = pd.DataFrame({
    "car_ID": sample_ids.values,
    "Фактична ціна": sample_y.values,
    "Прогноз": sample_pred,
    "Абсолютна помилка": np.abs(sample_y.values-sample_pred),
    "Абсолютна % помилка": np.abs((sample_y.values-sample_pred)/sample_y.values)*100
})
display(sample_results.round(2))
print("Середня абсолютна помилка для вибраних авто:", sample_results["Абсолютна помилка"].mean())

## 15. Висновки

Сформулювати підсумкові висновки за результатами роботи.

У висновках:
- порівняти якість усіх моделей;
- проаналізувати R², MAE, MSE і MAPE;
- зазначити, яка модель показала найкращі результати за отриманими метриками;
- прокоментувати величину похибок;
- звернути увагу на можливі ознаки перенавчання;
- коротко оцінити вплив попередньої обробки даних на результати моделювання.


In [ ]:
best = results_df.iloc[0]
print(f"За тестовою вибіркою найкраще значення R² показала модель {best['Модель']} (R²={best['R²']:.4f}).")
print(f"Для неї MAE={best['MAE']:.2f}, MSE={best['MSE']:.2f}, MAPE={best['MAPE (%)']:.2f}%.")
print("R² характеризує частку варіації цільової змінної, пояснену моделлю; MAE та MAPE відображають типову величину похибки, MSE сильніше штрафує великі помилки.")
print("Порівнюйте метрики разом: модель із найбільшим R² не обов'язково має найменші MAE/MSE/MAPE.")
print("Ознаки перенавчання остаточно оцінюються через порівняння train і test метрик; у цій роботі основне порівняння наведено для тесту.")
print("Викиди обмежувалися за IQR-межами train, числові ознаки трансформували Yeo–Johnson та масштабували, категоріальні кодували One-Hot Encoding. Це дає моделям узгоджену підготовку даних.")

## Загальні вимоги до виконання

- Не використовувати тестову вибірку для визначення меж викидів, кореляційного відбору ознак, навчання трансформерів, масштабування або кодування категорій.
- Усі параметри передобробки мають визначатися тільки за навчальною вибіркою.
- Фінальні метрики моделей обчислювати на тестовій вибірці.
- Не використовувати `car_ID` як предиктор.
- Не видаляти спостереження або ознаки без пояснення причини.
- До кожного основного етапу додати короткий змістовний висновок.
- У підсумкових висновках спиратися на отримані метрики та графіки.
